# Modèles locaux et agents IA 🤖

Dans ce tutoriel, nous allons apprendre à télécharger et exécuter un modèle local, puis à interagir avec lui en Python.


## Qu'est-ce qu'Ollama ? 🦙

Ollama est un outil qui permet d'exécuter des LLM sur votre propre machine. Il fournit une interface en ligne de commande simple et une API HTTP locale pour interagir avec les modèles.

Il a principalement été conçu pour les modèles texte-vers-texte, mais peut aussi exécuter d'autres types de modèles (texte-vers-image, texte-vers-audio, etc.).


## Installer Ollama 📦

Rendez-vous sur le [site officiel](https://ollama.com/download/).

Installez la version correspondant à votre système d'exploitation (macOS, Windows ou Linux). Vous devrez copier-coller une commande dans votre terminal (utilisez PowerShell sous Windows et un terminal sous macOS ou Linux).

Vérifiez que l'installation a fonctionné en exécutant :

```bash
ollama --version
```

Ollama doit fonctionner en arrière-plan. Pour vérifier, ouvrez `localhost:11434` dans votre navigateur : le message « Ollama is running » doit apparaître.

Voyons maintenant quels LLM sont disponibles :
https://ollama.com/library

Choisissons un modèle très léger, comme [FunctionGemma](https://ollama.com/library/functiongemma), qui devrait fonctionner sur la plupart des ordinateurs portables.

Pour télécharger le modèle, exécutez cette commande dans le terminal :

```bash
ollama pull functiongemma:270m
```

Le modèle est maintenant téléchargé et enregistré sur votre machine. Vérifiez-le avec :

```bash
ollama list
```


## Exécuter le modèle dans un terminal ▶️

Lançons le modèle avec :

```bash
ollama run functiongemma
```

Essayons de discuter avec lui ! Vous verrez qu'il n'est pas très doué pour répondre aux questions générales. C'est un petit modèle entraîné comme assistant de fonctions, et non comme agent conversationnel généraliste.

Pour quitter la conversation, tapez :
```bash
/bye
```


## Interagir avec le modèle en Python : l'API HTTP 🔌

Ollama est un **serveur web**. Tout programme capable d'envoyer une requête HTTP peut l'utiliser, y compris un programme Python avec la bibliothèque `requests`. C'est la meilleure façon de comprendre ce qui se passe en coulisses.

Commençons par vérifier que le serveur est accessible :


In [ ]:
import requests

BASE_URL = "http://localhost:11434" # Ollama default port

try:
    response = requests.get(BASE_URL, timeout=3)
    print(response.status_code, response.text)
except requests.ConnectionError:
    print("Ollama is not running. Start the Ollama app, or run `ollama serve` in a terminal.")

Le point de terminaison `/api/tags` liste les modèles installés, comme le fait `ollama list` :


In [ ]:
installed = requests.get(f"{BASE_URL}/api/tags").json()

for model in installed["models"]:
    print(f'{model["name"]:35s} {model["size"] / 1e9:5.1f} GB')

Vous pouvez également consulter l'API JSON depuis votre navigateur à l'adresse http://localhost:11434/api/tags


### 4.1 Générer une complétion ✍️

Le point de terminaison `/api/generate` reçoit une **consigne** (*prompt*) et renvoie la réponse du modèle. Avec `"stream": False`, nous recevons toute la réponse dans un seul objet JSON.


In [ ]:
MODEL = "functiongemma:270m"

payload = {
    "model": MODEL,
    "prompt": "Hello",
    "stream": False,
}

response = requests.post(f"{BASE_URL}/api/generate", json=payload, timeout=300)
# response.raise_for_status()
data = response.json()

print(data["response"])

Supposons que le programme ait répondu : « Hello! What can I help you with today? ».

Examinons la variable `data`, qui contient la réponse du modèle :

```python
{'model': 'functiongemma:270m',
 'created_at': '2026-11-18T17:52:49.383832848Z',
 'response': 'Hello! What can I help you with today?',
 'done': True,
 'done_reason': 'stop',
 'context': [2,
  105,
  2430,
  107,
  9259,
  106,
  236743,
  107,
  105,
  2028,
  107,
  9259,
  236888,
  2900,
  740,
  564,
  1601,
  611,
  607,
  3124,
  236881],
 'total_duration': 1376421445,
 'load_duration': 1090645823,
 'prompt_eval_count': 11,
 'prompt_eval_cached_count': 0,
 'prompt_eval_duration': 47761000,
 'eval_count': 11,
 'eval_duration': 236288000}
 ```

Le contexte est une liste de jetons, c'est-à-dire les fragments de texte vus par le modèle. Chaque jeton possède un identifiant, un texte décodé et un rôle (utilisateur ou modèle).

| Index | ID du jeton | Texte décodé / balise | Rôle |
| :--- | :--- | :--- | :--- |
| **0** | `2` | `<bos>` | Début de la séquence |
| **1** | `105` | `<start_of_turn>` | Ouvre le tour de parole |
| **2** | `2430` | `user` | Identifiant du rôle |
| **3** | `107` | `\n` | Séparateur de ligne |
| **4** | **`9259`** | **`hello`** | **Votre consigne** |
| **5** | `106` | `<end_of_turn>` | Ferme le tour utilisateur |
| **6** | `236743` | *(espace / jeton de formatage)* | Séparateur de modèle |
| **7** | `107` | `\n` | Séparateur de ligne |
| **8** | `105` | `<start_of_turn>` | Ouvre le tour de parole |
| **9** | `2028` | `model` | Identifiant du rôle |
| **10** | `107` | `\n` | Séparateur de ligne |
| **11** | **`9259`** | **`Hello`** | **Début de la réponse du modèle** |
| **12–20** | `236888` ... `236881` | `! What can I help you with today?` | Suite de la réponse (`236881` = `?`) |


In [ ]:
data

La réponse contient également des statistiques utiles. Les durées sont exprimées en nanosecondes.


In [ ]:
tokens_per_second = data["eval_count"] / (data["eval_duration"] / 1e9)

print(f'Tokens generated : {data["eval_count"]}')
print(f'Speed            : {tokens_per_second:.1f} tokens/s')
print(f'Model load time  : {data["load_duration"] / 1e9:.2f} s')

La sortie devrait ressembler à ceci :

```
Tokens generated : 11
Speed            : 46.6 tokens/s
Model load time  : 1.09 s
```


> **À retenir :** un modèle génère du texte **jeton par jeton** (un jeton peut être un mot ou une partie de mot). Une vitesse de 10 à 20 jetons par seconde est déjà confortable à lire.


## Créer une fonction personnalisée pour discuter avec le modèle 🛠️

Créons une fonction Python qui reçoit une liste de messages et renvoie la réponse du modèle. Chaque message est un dictionnaire comportant un champ `role` et un champ `content`. Les rôles possibles sont :

- **`system`** : une consigne système envoyée une seule fois au début de la conversation.
- **`user`** : un message envoyé par l'utilisateur.


In [ ]:
def chat(messages, **options):
    # Send a conversation to the model and return the text of its answer.
    payload = {
        "model": MODEL,
        "messages": messages,
        "stream": False,
        "options": options,
    }
    response = requests.post(f"{BASE_URL}/api/chat", json=payload, timeout=300)
    response.raise_for_status() # Raise an exception if the request failed
    return response.json()["message"]["content"]


messages = [
    {"role": "system", "content": "You are a concise teaching assistant. Answer in at most two sentences."},
    {"role": "user", "content": "What is a token in the context of language models?"},
]

answer = chat(messages)
print(answer)

**Ce modèle n'a pas de mémoire.** Le serveur ne se souvient pas des requêtes précédentes : pour qu'une question de suivi ait du sens, vous devez renvoyer toute la conversation, y compris la réponse précédente.


### 4.3 Contrôler la génération avec des options ⚙️

Le dictionnaire `options` modifie la façon dont le jeton suivant est choisi. Voici les options les plus utiles :

| Option | Signification |
|---|---|
| `temperature` | Aléatoire. `0` = sélection du jeton le plus probable (presque déterministe) ; une valeur plus élevée rend la réponse plus créative |
| `seed` | Fixe le générateur aléatoire afin de reproduire une réponse |
| `num_predict` | Nombre maximal de jetons à générer |
| `num_ctx` | Taille de la fenêtre de contexte (quantité de texte que le modèle peut lire) |

Posons plusieurs fois la même question avec différentes températures :


In [ ]:
question = [{"role": "user", "content": "Name 5 animals that can fly."}]

count = 0
for temperature in (0, 0, 1, 1, 2,2, 12, 10**20):
    print(f"temperature={temperature}:\n{chat(question, temperature=temperature)}")
    count += 1
    if count % 2 == 0: print("\n --- \n")

Avec `temperature=0`, vous devriez obtenir deux fois la même réponse. Avec une température élevée, les réponses varient.

À une température extrêmement élevée, le modèle échantillonne à partir d'une distribution de probabilités beaucoup plus uniforme.

Il privilégie alors moins fortement les jetons qui sont les plus cohérents avec le contexte.


### 4.4 Diffuser la réponse en continu 📡

Attendre une réponse complète peut sembler long. Avec `"stream": True`, le serveur envoie la réponse **morceau par morceau**, sous forme d'un objet JSON par ligne. C'est ainsi que les interfaces de chat affichent progressivement le texte.


In [ ]:
import json


def stream_rest(prompt):
    payload = {"model": MODEL, "prompt": prompt, "stream": True}
    with requests.post(f"{BASE_URL}/api/generate", json=payload, stream=True, timeout=300) as response:
        response.raise_for_status()
        for line in response.iter_lines():
            if not line:
                continue
            chunk = json.loads(line)
            print(chunk["response"], end="", flush=True)
            if chunk["done"]:
                break
    print()


stream_rest("Give three short reasons why people run language models locally.")

### 4.5 Un point de terminaison compatible avec OpenAI 🔌

Ollama propose également une API compatible avec le format OpenAI à l'adresse `/v1`. De nombreux outils et bibliothèques (y compris le paquet Python officiel `openai`, en définissant `base_url="http://localhost:11434/v1"`) peuvent ainsi utiliser votre modèle local comme remplacement direct d'un modèle hébergé.


In [ ]:
payload = {
    "model": MODEL,
    "messages": [{"role": "user", "content": "Say hello in French, in a few words."}],
}

response = requests.post(f"{BASE_URL}/v1/chat/completions", json=payload, timeout=300)
response.raise_for_status()
print(response.json()["choices"][0]["message"]["content"])

## 5. La bibliothèque Python `ollama` 🦙

L'API HTTP brute est simple, mais la bibliothèque officielle masque les détails (URL, JSON, diffusion en continu) et ajoute des fonctionnalités pratiques, comme les sorties structurées et les outils.

Installez-la dans l'environnement utilisé par ce notebook (avec `uv`, exécutez plutôt `uv add ollama` dans le projet) :


### Avec pip 📦
(Attention : la cellule suivante est en mode brut ; passez-la en mode Python pour l'exécuter.)


### Avec uv 📦

```bash
uv add ollama
```


### Un premier échange 💬


In [ ]:
import ollama

response = ollama.chat(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "Why is the sky blue? Answer in one sentence."
        }
    ],
    options={
        "temperature": 12
    }
)

print(response.message.content)

Même requête que précédemment, mais de façon plus concise. Pour diffuser la réponse en continu, ajoutez `stream=True` et parcourez les fragments :


In [ ]:
stream = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": "Name 5 animals that can fly."}],
    stream=True,
    options={
        "temperature": 12
    }
)

for chunk in stream:
    print(chunk.message.content, end="", flush=True)
print()

In [ ]:
from pydantic import BaseModel

class Book(BaseModel):
    title: str
    author: str
    year: int

text = """The Lord of the Rings is a fantasy novel written by J.R.R. Tolkien.
When Gandalf falls into the abyss with the Balrog, he says, "Fly, you fools!" to the Fellowship of the Ring.
The book was published in 1954 and is considered one of the greatest works of fantasy literature."""

response = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": f"Extract the book information from this text:\n{text}"}],
    format=Book.model_json_schema(),
    options={"temperature": 0},
)

print(response.message.content)

C'est très utile pour **extraire des informations** de documents, classer des textes ou alimenter une base de données. Les petits modèles peuvent se tromper sur les *valeurs* : validez toujours les résultats.


## Appeler une fonction avec FunctionGemma 🛠️

FunctionGemma est conçu pour choisir et appeler des fonctions, pas pour écrire leur code Python. Avec Ollama, transmettez une fonction Python comme outil ; sa signature et sa docstring décrivent l'outil. Cet exemple suit [l'exemple FunctionGemma d'Ollama](https://ollama.com/library/functiongemma).


In [ ]:
# Work in progress, not sure if it's working!

from ollama import chat

# 1. Define the Python function
def add(a, b):
    return a + b

# 2. Ask FunctionGemma to generate a function call
response = chat(
    model="functiongemma:270m",
    messages=[
        {"role": "user", "content": "Add 12 and 30"}
    ],
    tools=[
        {
            "type": "function",
            "function": {
                "name": "add",
                "description": "Add two numbers",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "a": {"type": "integer"},
                        "b": {"type": "integer"}
                    },
                    "required": ["a", "b"]
                }
            }
        }
    ]
)

# 3. Display the generated function call
print("Generated function call:")
print(response.message.tool_calls)

# 4. Execute the function with the generated arguments
if response.message.tool_calls:
    call = response.message.tool_calls[0]
    result = add(**call.function.arguments)
    print("\nResult:", result)
else:
    print("No function call generated.")